In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.1 KV caching

> 🎯 **Goal:** Cache the keys and values of past tokens so each new token costs constant work, and prove the cached output is identical to recomputing from scratch.

**Why this matters.** You trained a model in the earlier units. Now you have to *serve* it, which means generating one token at a time, fast and cheaply. The single biggest win in autoregressive generation is the KV cache: without it, serving long outputs is quadratically slow and you burn money re-deriving things you already knew.

**The intuition.** Imagine working a long math problem on a whiteboard. Every time you write a new line, you do not re-derive every line above it from the original premises, you just glance at the notes you already wrote and build on them. The KV cache is exactly those saved notes: the keys and values for tokens 1..t never change once computed, so you keep them and reuse them rather than recomputing them every step.

**The mechanics.** A quick vocabulary check. In attention, each token produces a **query** (what am I looking for), a **key** (what do I offer to others), and a **value** (the content I carry). To attend, the new query compares itself against the keys of all earlier tokens and mixes their values. Here is the key observation: as you generate token by token, the keys and values of *past* tokens are fixed. Naive decoding recomputes all of them at every step, so generating token t does O(t) work and the whole sequence is O(T squared). The **KV cache** is a small store that holds the past keys and values. At each step you compute the new token's K and V once, append them to the cache, and attend against the whole cache. New work per token drops to O(1) extra (you only process the one new token). The output must be bit-for-bit identical to full attention, only faster, and that parity is exactly what the code below checks.

In [ ]:
from pocketlm import scaled_dot_product_attention, KVCache, cached_step

torch.manual_seed(0)
B, H, T, hs = 1, 2, 6, 8
q = torch.randn(B, H, T, hs)
k = torch.randn(B, H, T, hs)
v = torch.randn(B, H, T, hs)
full, _ = scaled_dot_product_attention(q, k, v, causal=True)
cache = KVCache()
outs = [cached_step(q[:, :, t:t + 1], k[:, :, t:t + 1], v[:, :, t:t + 1], cache)
        for t in range(T)]
incremental = torch.cat(outs, dim=2)
print("cached == full attention:", torch.allclose(incremental, full, atol=1e-5))

**What just happened.** We computed full causal attention over all 6 tokens at once, then fed the same tokens one at a time through `cached_step`, appending to a `KVCache` each step. The printed `True` says the two results match within floating-point tolerance. That parity is the whole point: the cache is a pure performance optimization, it changes nothing about the answer. You traded a little memory (storing past K and V) for a large compute saving.

⚠️ **Common pitfalls**
- Forgetting the causal mask. Token t must still only see tokens 1..t. The cache holds the past, but you must not let a token peek at the future.
- Caching K and V but recomputing the query over the whole sequence. Only the *new* token needs a query each step, that is where the speedup comes from.
- Never resetting the cache between independent prompts, which silently leaks one request's context into the next.

🏋️ **Try it yourself.** Bump `T` to 32 and add a second test that times the full-attention path versus the cached loop (use `time.perf_counter` around each). The parity assert should still pass, and you should see the cached loop's per-token cost stay roughly flat while a from-scratch recompute would grow with length.

In [ ]:
assert torch.allclose(incremental, full, atol=1e-5)